In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from training.config import MODEL_NAME

In [8]:
dataset=load_dataset("csv",data_files="data/summaries.csv")
split_ds=dataset['train'].train_test_split(test_size=0.1,seed=42)

In [9]:
split_ds=split_ds.remove_columns(column_names=['policy_words','summary_words'])
split_ds

DatasetDict({
    train: Dataset({
        features: ['Summary', 'Policy'],
        num_rows: 4219
    })
    test: Dataset({
        features: ['Summary', 'Policy'],
        num_rows: 469
    })
})

In [10]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

In [18]:
PREFIX="summarize privacy policy: "
def preprocess_function(example):
    
    inputs=[PREFIX+text for text in example["Policy"]]
    
    model_ips=tokenizer(
        inputs,
        max_length=1024,
        truncation=True
    )
    
    labels=tokenizer(
        text_target=example["Summary"],
        max_length=128,
        truncation=True
    )
    
    model_ips["labels"]=labels['input_ids']
    return model_ips

In [20]:
tokenized_ds=split_ds.map(preprocess_function,batched=True)

Map:   0%|          | 0/4219 [00:00<?, ? examples/s]

Map:   0%|          | 0/469 [00:00<?, ? examples/s]

In [24]:
tokenized_ds=tokenized_ds.remove_columns(split_ds['train'].column_names)

In [29]:
tokenized_ds.save_to_disk("data/processed_ds")

Saving the dataset (0/1 shards):   0%|          | 0/4219 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/469 [00:00<?, ? examples/s]